In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
delivery_time_path = os.path.join(path, 'Q1_data.csv')
df_delivery_time = pd.read_csv(delivery_time_path)

print(f"Dataset shape: {df_delivery_time.shape}")
df_delivery_time.head()

In [ ]:
# Task 2: Write your code here:(head)
# Analyze missing values
missing_percentage = (df_delivery_time.isnull().sum() / len(df_delivery_time)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)
df_delivery_time.head()

In [ ]:
# Task 3: Display dataset information using info():
df_delivery_time.info()

In [ ]:
# Task 4(Show statistical description using (describe): Write your code here:
df_delivery_time.describe()

In [ ]:
# 5-Plot the target distribution (delivery_time)

# Is the target imbalanced?
def check_target_imbalance(df_delivery_time, target_column):
  print("Target Distribution:")


  df_delivery_time[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df_delivery_time, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:

# What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_delivery_time, "Order_ID")

In [ ]:
# Task 2: Write your code here:
#  Do we have missing values?
def check_missing_values(df_delivery_time):
  missing_values = df_delivery_time.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_delivery_time)

In [ ]:
# Task 3: Write your code here:
#  Check for duplicates
def check_duplicates(df_delivery_time):

  #TODO: get duplicated data using pandas
  duplicates = df_delivery_time.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_delivery_time.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_delivery_time)

In [ ]:
#  Do we have categorical columns?
categorical_cols = df_delivery_time.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Pick only the numerical columns, NOT the target
numerical_cols = df_delivery_time.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = MinMaxScaler()

# scale the `numerical_cols`
df_delivery_time[numerical_cols] = scaler.fit_transform(df_delivery_time[numerical_cols])

df_delivery_time.head()

In [ ]:
# Task 4: Encode categorical variables if needed (Bonus if used One Hot Encoding)
# Do we have different scales in the data?
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# Encode the target column
df_delivery_time['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type'] = le.fit_transform(df_delivery_time['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type'])

df_delivery_time
df_delivery_time.describe()

In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import MinMaxScaler

features = df_delivery_time.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = MinMaxScaler()
df_delivery_time[features] = scaler.fit_transform(df_delivery_time[features])
df_delivery_time.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df_delivery_time.drop("Delivery_Time", axis=1).astype(float)
y = df_delivery_time['Delivery_Time'].astype(float)



In [ ]:
# Task 2,3,4,5: Write your code here:
# Mean Squared Error in NumPy
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)
####
  def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

In [ ]:
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
# Storage for linear regression results for each fold
lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_mse.append(mse)
  lr_rmse.append(rmse)
  lr_r2.append(r2)

In [ ]:
#5-Print the averaged score across all folds

average_losses = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: